In [ ]:
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.family"] = "monospace"

In [ ]:
model_coa_mean_2 = pd.read_csv("../data/model_coa_mean_barplot.csv").set_index("model")
model_coa_mean_2_future = pd.read_csv(
    "../data/model_coa_mean_future_barplot.csv"
).set_index("model")

model_coa_mean_2 = model_coa_mean_2.sort_values("coa_event")
model_order_2 = model_coa_mean_2.index
model_coa_mean_2_future = model_coa_mean_2_future.reindex(
    model_order_2, fill_value=np.nan
)

obs_idx1 = np.where(model_order_2 == "HadISST")[0]
obs_idx2 = np.where(model_order_2 == "ERSSTv5")[0]


In [ ]:
red_models = []
green_models = []
orange_models = []

if (
    model_coa_mean_2.loc["ERSSTv5"].coa_event
    <= model_coa_mean_2.loc["HadISST"].coa_event
):
    min_obs, max_obs = "ERSSTv5", "HadISST"
else:
    min_obs, max_obs = "HadISST", "ERSSTv5"
min_thresh = model_coa_mean_2.loc[min_obs, "coa_q25"]
max_thresh = model_coa_mean_2.loc[max_obs, "coa_q75"]
if round(model_coa_mean_2.loc["ERSSTv5", "coa_event"], 1) == round(
    model_coa_mean_2.loc["HadISST", "coa_event"], 1
):
    min_thresh = model_coa_mean_2.loc["HadISST", "coa_q25"]

above_mask = model_coa_mean_2["coa_event"] > max_thresh
below_mask = model_coa_mean_2["coa_event"] < min_thresh

colors = []
for i in range(len(model_coa_mean_2)):
    if model_order_2[i] in ("ERSSTv5", "HadISST"):
        colors.append("blue")
    elif below_mask.iloc[i]:
        colors.append("orange")
        orange_models.append(model_order_2[i])
    elif above_mask.iloc[i]:
        colors.append("red")
        red_models.append(model_order_2[i])
    else:
        colors.append("green")
        green_models.append(model_order_2[i])

fig, ax = plt.subplots(figsize=(15, 5), dpi=300)
_lower = (
    (model_coa_mean_2["coa_event"] - model_coa_mean_2["coa_q25"])
    .clip(lower=0)
    .values
)
_upper = (
    (model_coa_mean_2["coa_q75"] - model_coa_mean_2["coa_event"])
    .clip(lower=0)
    .values
)
model_coa_mean_2.plot(
    kind="bar",
    y="coa_event",
    yerr=np.array([_lower, _upper]),
    ax=ax,
    color=colors,
    width=0.7,
    legend=False,
)

_lower_f = (
    (
        model_coa_mean_2_future["coa_event_future"]
        - model_coa_mean_2_future["coa_q25_future"]
    )
    .clip(lower=0)
    .fillna(0)
    .values
)
_upper_f = (
    (
        model_coa_mean_2_future["coa_q75_future"]
        - model_coa_mean_2_future["coa_event_future"]
    )
    .clip(lower=0)
    .fillna(0)
    .values
)
model_coa_mean_2_future.plot(
    kind="bar",
    y="coa_event_future",
    yerr=np.array([_lower_f, _upper_f]),
    ax=ax,
    color="grey",
    edgecolor="black",
    width=0.7,
    legend=False,
    alpha=0.5,
)

ax.axhline(min_thresh, ls="--", lw=0.5, c="k")
ax.axhline(max_thresh, ls="--", lw=0.5, c="k")

ax.grid(ls="dotted")

ax.text(
    0.02,
    0.75,
    "Models with # of COA events above observed",
    verticalalignment="bottom",
    horizontalalignment="left",
    color="red",
    fontsize=14,
    transform=ax.transAxes,
)
ax.text(
    0.02,
    0.80,
    "Models  with # of COA events within observed",
    verticalalignment="bottom",
    horizontalalignment="left",
    color="green",
    fontsize=14,
    transform=ax.transAxes,
)
ax.text(
    0.02,
    0.85,
    "Models  with # of COA events below observed",
    verticalalignment="bottom",
    horizontalalignment="left",
    color="orange",
    fontsize=14,
    transform=ax.transAxes,
)
ax.text(
    0.02,
    0.90,
    "Reanalysis",
    verticalalignment="bottom",
    horizontalalignment="left",
    color="blue",
    fontsize=14,
    transform=ax.transAxes,
)

ax.text(
    0.02,
    0.95,
    "Future projections",
    verticalalignment="bottom",
    horizontalalignment="left",
    color="grey",
    fontsize=14,
    transform=ax.transAxes,
    path_effects=[path_effects.withStroke(linewidth=0.5, foreground="black")],
)

for label in ax.get_xticklabels():
    if label.get_text() in ["ERSSTv5", "HadISST"]:
        label.set_color("blue")
        label.set_fontweight("bold")

ax.set_yticks(np.arange(0, 12.1, 2))
ax.set_ylim(0, 14)
ax.set_xlabel("Model")
ax.set_ylabel("# of coastal El Niño events per century")
